In [ ]:
import torch

In [ ]:
with open('names.txt', 'r', encoding='UTF-8') as f:
    words = f.read().splitlines()

In [ ]:
# all characters in the dataset
chars = ['.'] + sorted(list(set(''.join(words))))
# string to integer encoder 
stoi = {ch:i for i, ch in enumerate(chars)}
# inverse mapping
itos = {i:ch for ch, i in stoi.items()}

In [ ]:
# Create the training set of bigrams (x, y)
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        xs.append(idx1)
        ys.append(idx2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

# Check if MPS is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Initialize the network
g = torch.Generator(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True, device=device)

In [ ]:
# We can augment the loss function to incentivize model smoothing by
# encouraging the loss to bring all weights closer to 0.
# when all weights are 0, exp(0) -> 1 -> probs -> uniform.
# We can do this by tacking on an additional term (W**2).sum() which
# adds contribution the loss function the further a value is from 0
# We can scale this value with some constant (regularization strength) 
# to lower/raise its impact on the loss function
# This is exactly the same as adding a constant when model smoothing
# the counts matrix. 
# This is known as loss regularization
(W**2).mean()

In [ ]:
import torch.nn.functional as F
# gradient descent
for k in range(100):

    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float().to(device=device)
    logits = xenc @ W
    # These two steps below are a softmax, which takes in a distribution
    # of values (like from the normal dist) and turns it into a prob distribution
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True) # make each row a prob distribution
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    print(loss.item())

    # backward pass
    W.grad = None
    loss.backward()

    # Update gradient
    W.data += -50 * W.grad


In [ ]:
# Sampling from the neural network
g = torch.Generator(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        # forward pass
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float().to(device=device)
        logits = xenc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims=True)
        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if itos[ix] == '.':
            out.pop()
            break
    print(''.join(out))